In [1]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster import hierarchy

#from models import CLIPModel, CLIPModel_ViT, CLIPModel_ViT_L, CLIPModel_CLIP, CLIPModel_resnet152, CLIPModel_resnet101
#from dataset import CLIPDataset
import scanpy as sc
from torch.utils.data import DataLoader

import os
import numpy as np
import scanpy as sc
import pandas as pd

%run /sibcb1/chenluonanlab8/zoujiawei/Final_version_with_foundation_model/model/impute_lung.py

In [2]:
#print the current scanpy version
print(sc.__version__)

1.10.3


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import List, Callable, Tuple

#from transformers import BertModel, BertConfig
#from modules import ImageEncoder, ImageEncoder_ViT, ImageEncoder_ViT_L, ImageEncoder_CLIP, ImageEncoder_resnet101, ImageEncoder_resnet152
import math

class CFG:
    projection_dim = 256
    dropout = 0.1
    temperature = 0.07
    image_embedding = 384

# Define ProjectionHead class
class ProjectionHead(nn.Module):
    def __init__(
        self,
        embedding_dim,
        projection_dim=CFG.projection_dim,
        dropout=CFG.dropout,
        eps=1e-06  # 修改后的 eps 参数
    ):
        super().__init__()
        self.projection = nn.Linear(embedding_dim, projection_dim)
        self.gelu = nn.GELU()
        self.fc = nn.Linear(projection_dim, projection_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(projection_dim, eps=eps)  # 在这里设置 eps
    
    def forward(self, x):
        projected = self.projection(x)
        x = self.gelu(projected)
        x = self.fc(x)
        x = self.dropout(x)
        x = x + projected
        x = self.layer_norm(x)
        return x



In [4]:
#torch.cuda.empty_cache()
import sys
sys.path.append('/sibcb1/chenluonanlab8/zoujiawei/HIPT/HIPT_4K')

from hipt_4k import HIPT_4K
from hipt_model_utils import get_vit256, get_vit4k, eval_transforms
from hipt_heatmap_utils import *
light_jet = cmap_map(lambda x: x/2 + 0.5, matplotlib.cm.jet)

pretrained_weights256 = '../Checkpoints/vit256_small_dino.pth'
#pretrained_weights4k = '../Checkpoints/vit4k_xs_dino.pth'
device256 = torch.device('cuda')
#device4k = torch.device('cpu')

### ViT_256 + ViT_4K loaded independently (used for Attention Heatmaps)
model256 = get_vit256(pretrained_weights=pretrained_weights256, device=device256)
#model4k = get_vit4k(pretrained_weights=pretrained_weights4k, device=device4k)

# 首先冻结 model256 的所有参数
for param in model256.parameters():
    param.requires_grad = False

# 解冻最后一个 block 的所有参数
for param in model256.blocks[-1].parameters():
    param.requires_grad = True

for param in model256.norm.parameters():
    param.requires_grad = True

In [5]:
ngenes = 2000

def cross_entropy(preds, targets, reduction='none'):
    log_softmax = nn.LogSoftmax(dim=-1)
    loss = (-targets * log_softmax(preds)).sum(1)
    if reduction == "none":
        return loss
    elif reduction == "mean":
        return loss.mean()

class HIPT_CLIP_Model(nn.Module):
    def __init__(self, temperature=1, image_embedding=384, spot_embedding=ngenes):
        super().__init__()
        # 使用 VisionTransformer 作为新的 image_encoder
        self.image_encoder = model256
        #self.image_projection = ProjectionHead(embedding_dim=image_embedding)  # aka the input dim, 768 for VIT
        self.spot_projection = ProjectionHead(embedding_dim=spot_embedding,projection_dim=384)    # 3467 shared hvgs
        self.temperature = nn.Parameter(torch.ones([]) * np.log(1 / temperature))

    def forward(self, batch):
        # Getting Image and spot Features
        #print(batch["image"].shape)
        image_features = self.image_encoder(batch["image"])
        #print(image_features.shape)
        spot_features = batch["reduced_expression"]

        # Getting Image and Spot Embeddings (with same dimension) 
        #image_embeddings = self.image_projection(image_features)
        image_embeddings = image_features
        spot_embeddings = self.spot_projection(spot_features)

        #L2 normalization
        norm_image = image_embeddings.norm(p=2, dim=1, keepdim=True) 
        image_embeddings = image_embeddings/norm_image

        norm_spot = spot_embeddings.norm(p=2, dim=1, keepdim=True) 

        spot_embeddings = spot_embeddings/norm_spot


        # Calculating the Loss the logit prob for gradcam
        logits = (spot_embeddings @ image_embeddings.T) / self.temperature.exp()
        probabilities = F.softmax(logits.T, dim=1)
        
        return probabilities

    def encode_image(self, image):
        return self.image_encoder(image)

    def encode_spot(self, spot):
        return self.spot_projection(spot)

In [6]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import cv2
import pandas as pd
import tifffile  # 用于读取 TIFF 图像


class augmentDataset1(torch.utils.data.Dataset):
    def __init__(self, image_path, valid_centers_path, augment_factor=2):
        """
        初始化数据集。

        Args:
            image_path (str): TIFF 图像文件的路径。
            valid_centers_path (str): 包含有效中心点坐标的文本文件路径。
            augment_factor (int): 数据增强因子。
        """
        self.image_path = image_path
        self.valid_centers_path = valid_centers_path
        self.augment_factor = augment_factor

        # 读取 TIFF 图像
        try:
            #self.whole_image = cv2.imread(self.image_path, cv2.IMREAD_UNCHANGED) #opencv读取tif
            self.whole_image = tifffile.imread(self.image_path) #tifffile读取tif
            if self.whole_image is None:
                raise ValueError(f"Image could not be loaded from {self.image_path}.")
            if self.whole_image.ndim == 2:  # 如果是灰度图，转换为 RGB
                self.whole_image = cv2.cvtColor(self.whole_image, cv2.COLOR_GRAY2RGB)
            elif self.whole_image.shape[2] == 4:  # 如果是 RGBA, 去掉 Alpha 通道
                self.whole_image = self.whole_image[:, :, :3]
                self.whole_image = cv2.cvtColor(self.whole_image, cv2.COLOR_RGBA2RGB)
            if self.whole_image.dtype != np.uint8: #转换为uint8
                if self.whole_image.dtype == np.uint16:
                  self.whole_image = (self.whole_image / 256).astype(np.uint8) #16位转8位
                else:
                    self.whole_image = self.whole_image.astype(np.uint8)
        except Exception as e:
            raise ValueError(f"Error loading image from {self.image_path}: {e}")

        # 读取有效中心点坐标
        try:
            self.valid_centers = pd.read_csv(self.valid_centers_path, sep=',', header=None)  # 假设是制表符分隔
            if self.valid_centers.shape[1] < 2:  # 确保至少有两列
                raise ValueError(f"valid_centers file should have at least two columns (v1, v2).")
            self.valid_centers = self.valid_centers.iloc[:, :2].values  # 取前两列, 转为 NumPy 数组
        except Exception as e:
            raise ValueError(f"Error loading valid centers from {self.valid_centers_path}: {e}")

        print("Finished loading image and valid centers.")

        self.augmentations = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(degrees=(0, 360)),
            transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
            transforms.RandomResizedCrop(size=(256, 256), scale=(0.8, 1.0)),
        ])

    def transform(self, image):
        image = Image.fromarray(image)
        image = self.augmentations(image)
        return np.asarray(image)

    def __getitem__(self, idx):
        while True:
            actual_idx = idx % len(self.valid_centers)
            augment_idx = idx // len(self.valid_centers)

            v1, v2 = self.valid_centers[actual_idx]
            v1, v2 = int(v1), int(v2)  # 确保是整数

            # 提取图像块
            image_patch = self.whole_image[(v1 - 128):(v1 + 128), (v2 - 128):(v2 + 128)]

            if image_patch.shape[0] != 256 or image_patch.shape[1] != 256:
                idx += 1
                continue

            if augment_idx > 0:
                image_patch = self.transform(image_patch)

            #image_patch_rgb = cv2.cvtColor(image_patch, cv2.COLOR_BGR2RGB) #如果一开始就是RGB, 不需要转换
            image_patch_pil = Image.fromarray(image_patch)
            t = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
            ])
            image = t(image_patch_pil)

            item = {
                'image': image,
                'spatial_coords': torch.tensor([v1, v2]).float()
            }
            return item

    def __len__(self):
        return len(self.valid_centers) * self.augment_factor

In [7]:
import torch
#import torchvision.transforms as transforms  # 不需要
#from PIL import Image  # 不需要
import numpy as np
#import cv2  # 不需要
#import pandas as pd  # 不需要
from scipy.sparse import csr_matrix  # 导入 csr_matrix


class augmentDataset2(torch.utils.data.Dataset):
    def __init__(self, scaled_matrix, augment_factor=1):
        """
        初始化数据集。

        Args:
            scaled_matrix (csr_matrix): 缩放后的表达矩阵 (cell x features)。
            augment_factor (int):  数据增强因子 (这里没有图像增强，所以可能不需要).
        """
        self.scaled_matrix = scaled_matrix
        self.augment_factor = augment_factor

        # 检查 scaled_matrix 的类型和维度
        if not isinstance(self.scaled_matrix, csr_matrix):
            raise TypeError("scaled_matrix must be a csr_matrix object")
        if self.scaled_matrix.ndim != 2:
            raise ValueError("scaled_matrix must be a 2D csr_matrix with shape (13200, 2000)")

        print("Finished loading scaled matrix.")

    def __getitem__(self, idx):
        actual_idx = idx % self.scaled_matrix.shape[0]
        item = {}

        # 从稀疏矩阵中获取数据并转换为 PyTorch Tensor
        rd_ex = self.scaled_matrix[actual_idx, :].toarray()  # 使用 toarray() 转换为 NumPy 数组
        rd_ex = rd_ex.squeeze()  # 去除多余的维度 (结果应该是一个一维数组)
        item['reduced_expression'] = torch.tensor(rd_ex).float()

        return item

    def __len__(self):
        return self.scaled_matrix.shape[0]

In [8]:
import anndata as ad
import scanpy as sc
import pandas as pd

# 读取 .h5ad 文件
adata = ad.read_h5ad("/sibcb1/chenluonanlab8/zoujiawei/data_lung/10x/10x/VHD/lung_ref_temp_300.h5ad")

In [9]:
scaled_matrix = adata.X
scaled_matrix.shape


(13242, 2000)

In [10]:
ref_dataset = augmentDataset2(scaled_matrix)

# 获取一个样本
sample = ref_dataset[0]
reduced_expression = sample['reduced_expression']

print(f"Reduced expression shape: {reduced_expression.shape}")  # 输出: Reduced expression shape: torch.Size([2000])


# 使用 DataLoader (可选)
from torch.utils.data import DataLoader

dataloader = DataLoader(ref_dataset, batch_size=32, shuffle=False)

Finished loading scaled matrix.
Reduced expression shape: torch.Size([2000])


In [11]:
pretrained_weights_path = "/sibcb1/chenluonanlab8/zoujiawei/data_lung/10x/10x/VHD/model/HIPT_no_projection_200_gene_threshold.pt"
pretrained_state_dict = torch.load(pretrained_weights_path, map_location=torch.device('cpu'))

# Initialize the CLIP model
device = "cuda"
hipt_model = HIPT_CLIP_Model().to(device)
hipt_model.load_state_dict(pretrained_state_dict)
hipt_model.eval()
model = hipt_model

In [12]:

def get_image_embeddings(model,test_loader):
    test_loader = test_loader
    # model = CLIPModel().cuda()
    model.eval()

    print("Finished loading model")
    
    test_image_embeddings = []
    with torch.no_grad():
        for batch in tqdm(test_loader):
            image_features = model.encode_image(batch["image"].cuda())
            #image_embeddings = model.image_projection(image_features)
            test_image_embeddings.append(image_features)
    
    return torch.cat(test_image_embeddings)


def get_spot_embeddings(model,test_loader):
    test_loader = test_loader
    # model = CLIPModel().cuda()

    model.eval()

    print("Finished loading model")

    spot_embeddings = []
    with torch.no_grad():
        for batch in tqdm(test_loader):
            # spot_features = model.spot_encoder(batch["reduced_expression"].cuda()) 
            # spot_embeddings = model.spot_projection(spot_features)
            spot_embeddings.append(model.encode_spot(batch["reduced_expression"].cuda()))
    return torch.cat(spot_embeddings)


#2265x256, 2277x256
def find_matches(spot_embeddings, query_embeddings, top_k=1):
    #find the closest matches 
    spot_embeddings = torch.tensor(spot_embeddings)
    query_embeddings = torch.tensor(query_embeddings)
    query_embeddings = F.normalize(query_embeddings, p=2, dim=-1)
    spot_embeddings = F.normalize(spot_embeddings, p=2, dim=-1)
    dot_similarity = query_embeddings @ spot_embeddings.T   #2277x2265
    print(dot_similarity.shape)
    _, indices = torch.topk(dot_similarity.squeeze(0), k=top_k)
    
    return indices.cpu().numpy()

In [13]:
#img_embeddings_all = get_image_embeddings(model,dataset_img)
spot_embeddings_all = get_spot_embeddings(model,dataloader)

Finished loading model


100%|██████████| 414/414 [00:01<00:00, 215.82it/s]


### run match 

In [14]:
save_path = '/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/Results/153074A1A2A3-PPA60-APA35-MPA5/'
valid_centers_path = '/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/valid_centers_standardized_153074A1A2A3-PPA60-APA35-MPA5.txt'
image_path = '/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/standardized_153074A1A2A3-PPA60-APA35-MPA5.tif'
augment_factor = 1

# 创建数据集实例
dataset_img = augmentDataset1(image_path, valid_centers_path, augment_factor)
from torch.utils.data import DataLoader

dataloader_img = DataLoader(dataset_img, batch_size=32, shuffle=False)

Finished loading image and valid centers.


In [15]:
import os


# 检查路径是否存在
if not os.path.exists(save_path):
    # 如果不存在，则创建路径（包括所有父目录）
    os.makedirs(save_path)
    print(f"Path created: {save_path}")
else:
    print(f"Path already exists: {save_path}")

Path already exists: /sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/Results/153074A1A2A3-PPA60-APA35-MPA5/


In [16]:
img_embeddings_all = get_image_embeddings(model,dataloader_img)

Finished loading model


  0%|          | 0/54 [00:00<?, ?it/s]/sibcb1/chenluonanlab8/zoujiawei/miniconda3/envs/CarfHe/lib/python3.9/site-packages/torch/nn/functional.py:3060: UserWarning: Default upsampling behavior when mode=bicubic is changed to align_corners=False since 0.4.0. Please specify align_corners=True if the old behavior is desired. See the documentation of nn.Upsample for details.
  warnings.warn("Default upsampling behavior when mode={} is changed "
/sibcb1/chenluonanlab8/zoujiawei/miniconda3/envs/CarfHe/lib/python3.9/site-packages/torch/nn/functional.py:3103: UserWarning: The default behavior for interpolate/upsample with float scale_factor changed in 1.6.0 to align with other frameworks/libraries, and now uses scale_factor directly, instead of relying on the computed output size. If you wish to restore the old behavior, please set recompute_scale_factor=True. See the documentation of nn.Upsample for details. 
  warnings.warn("The default behavior for interpolate/upsample with float scale_facto

In [17]:
#outputs:
#data sizes: (3467, 2378) (3467, 2349) (3467, 2277) (3467, 2265)

img_embeddings_all = img_embeddings_all.cpu().numpy()
spot_embeddings_all = spot_embeddings_all.cpu().numpy()
print(img_embeddings_all.shape)
print(spot_embeddings_all.shape)

if not os.path.exists(save_path):
    os.makedirs(save_path)

if not os.path.exists(save_path):
    os.makedirs(save_path)


np.save(save_path + "img_embeddings_" + ".npy", img_embeddings_all.T)
np.save(save_path + "spot_embeddings_" + ".npy", spot_embeddings_all.T)



(1717, 384)
(13242, 384)


In [18]:
#save_path = "results/brca/"
spot_embeddings1 = np.load(save_path + "spot_embeddings_.npy")

image_embeddings1 = np.load(save_path + "img_embeddings_.npy")



#expression_key = np.concatenate([spot_expression1,spot_expression2,spot_expression3], axis = 1)



In [19]:
expression_gt = scaled_matrix

In [20]:
gene_names = adata.var_names

In [21]:
#query
image_query = image_embeddings1
spot_key = spot_embeddings1
expression_key = expression_gt
#expression_gt = spot_expression1

In [22]:
expression_key

<13242x2000 sparse matrix of type '<class 'numpy.float64'>'
	with 560419 stored elements in Compressed Sparse Row format>

In [23]:
import numpy as np
import torch
from scipy.sparse import csr_matrix

# ... (find_matches 函数) ...
def find_matches(spot_key, image_query, top_k=5):
    spot_key_tensor = torch.tensor(spot_key)
    image_query_tensor = torch.tensor(image_query)
    distances = torch.cdist(image_query_tensor, spot_key_tensor)
    _, indices = torch.topk(distances, top_k, dim=1, largest=False)
    return indices.numpy()

method = "average"

if image_query.shape[1] != 384:
    image_query = image_query.T
print("image query shape: ", image_query.shape)

if spot_key.shape[1] != 384:
    spot_key = spot_key.T
print("spot_key shape: ", spot_key.shape)

if expression_key.shape[0] != spot_key.shape[0]:
    expression_key = expression_key.T
print("expression_key shape: ", expression_key.shape)

# 转换为密集数组 (如果内存允许)
if isinstance(expression_key, csr_matrix): # 或者  issparse(expression_key)
    expression_key = expression_key.toarray()

if method == "average":
    print("finding matches, using average of top k expressions")
    top_k = 5
    indices = find_matches(spot_key, image_query, top_k=top_k)
    print("indices shape:", indices.shape)

    matched_spot_embeddings_pred = np.zeros((indices.shape[0], spot_key.shape[1]))
    matched_spot_expression_pred = np.zeros((indices.shape[0], expression_key.shape[1]))

    for i in range(indices.shape[0]):
        matched_spot_embeddings_pred[i, :] = np.average(spot_key[indices[i, :], :], axis=0)
        matched_spot_expression_pred[i, :] = np.average(expression_key[indices[i, :], :], axis=0)

    print("matched spot embeddings pred shape: ", matched_spot_embeddings_pred.shape)
    print("matched spot expression pred shape: ", matched_spot_expression_pred.shape)

pred = matched_spot_expression_pred
print("Predicted expression shape:", pred.shape)

image query shape:  (1717, 384)
spot_key shape:  (13242, 384)
expression_key shape:  (13242, 2000)
finding matches, using average of top k expressions
indices shape: (1717, 5)
matched spot embeddings pred shape:  (1717, 384)
matched spot expression pred shape:  (1717, 2000)
Predicted expression shape: (1717, 2000)


In [24]:
if save_path != "":
    np.save(save_path + "matched_spot_embeddings_pred.npy", matched_spot_embeddings_pred.T)
    np.save(save_path + "matched_spot_expression_pred.npy", matched_spot_expression_pred.T)

## adata produce

In [25]:
import pandas as pd
import torch

## spatial location of cells/spots

spatial = pd.read_csv(valid_centers_path, sep=',', header=None)



spatial.head()



,0,1
0,128,128
1,128,384
2,128,640
3,128,896
4,128,1152


In [26]:
v1 = spatial.iloc[0:,0]
v2 = spatial.iloc[0:,1]

## marker gene heatmap

In [27]:

#"/sibcb1/chenluonanlab8/zoujiawei/Carhe/brcadata/st/combined/sp/H1.csv"
gene_names 


Index(['PERM1', 'HES4', 'SCNN1D', 'TMEM52', 'HES5', 'MEGF6', 'AJAP1', 'TAS1R1',
       'ERRFI1', 'TMEM201',
       ...
       'FMR1NB', 'MAGEA8', 'CNGA2', 'PNMA6E', 'ATP2B3', 'DUSP9', 'F8A1',
       'FUNDC2', 'PCDH11Y', 'MT-ND6'],
      dtype='object', length=2000)

In [28]:
matched_spot_expression_pred_1 = np.load(save_path+"matched_spot_expression_pred.npy")

In [29]:
# Convert to DataFrames

matched_spot_expression_pred_1_df = pd.DataFrame(matched_spot_expression_pred_1)

expression_data = pd.DataFrame(matched_spot_expression_pred_1.T, columns=gene_names[:ngenes]) 


In [30]:
expression_data.shape

(1717, 2000)

In [31]:
spatial.shape

(1717, 2)

In [32]:
save_path

'/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/Results/153074A1A2A3-PPA60-APA35-MPA5/'

In [33]:
# 选择要可视化的基因
gene_to_plot = gene_names[0]
print(gene_to_plot)
# 例如 HAL [ "HAL", "CYP3A4", "VWF", "SOX9", "KRT7", "ANXA4", "ACTA2", "DCN"] #markers from macparland paper

PERM1


In [34]:
#TLS genes
tls_genes = [
                'ANLN', 'ASPM', 'CDCA4', 'ERRFI1', 'FURIN', 'GOLGA8A', 'ITGA6',
                'JAG1', 'LRP12', 'MAFF', 'MRPS17', 'PLK1', 'PNP', 'PPP1R13L',
                'PRKCA', 'PTTG1', 'PYGB', 'RPP25', 'SCPEP1', 'SLC46A3', 'SNX7',
                'TPBG', 'XBP1'
            ]
common_genes = np.intersect1d(gene_names, tls_genes)

# 输出结果
print(common_genes)

['CDCA4' 'ERRFI1' 'JAG1' 'XBP1']


In [35]:
gene_to_plot = "CXCL13"

In [36]:
expression_data = pd.DataFrame(matched_spot_expression_pred_1.T, columns=gene_names) 

In [37]:
image_path

'/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/standardized_153074A1A2A3-PPA60-APA35-MPA5.tif'

In [38]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import Normalize
import matplotlib.cm as cm
# Load the original image
original_image_rgb = cv2.imread(image_path)
if original_image_rgb is None:
    raise FileNotFoundError(f"Could not load image from path: {image_path}")

# Create a thumbnail by resizing the original image
scaling_factor = 0.1  # Adjust as needed (e.g., 0.1 for 10%)
thumbnail_width = int(original_image_rgb.shape[1] * scaling_factor)
thumbnail_height = int(original_image_rgb.shape[0] * scaling_factor)
thumbnail = cv2.resize(original_image_rgb, (thumbnail_width, thumbnail_height))

# Ensure v1 and v2 are numpy arrays and scale coordinates
v1 = np.array(v1, dtype=float)
v2 = np.array(v2, dtype=float)
v1_scaled = v1 * scaling_factor
v2_scaled = v2 * scaling_factor

# Create the base coordinate DataFrame
coord_data = pd.DataFrame({
    'X': v1_scaled,
    'Y': v2_scaled
})

# Combine coordinates and ALL expression data
# Assumes index alignment between coord_data and expression_data
plot_data_base = pd.concat([coord_data, expression_data], axis=1)

# --- Loop through each gene in common_genes ---

print(f"Starting plot generation for {len(common_genes)} genes...")

for gene_to_plot in common_genes:
    print(f"  Processing: {gene_to_plot}")

    # Check if the gene exists in the DataFrame columns
    if gene_to_plot not in plot_data_base.columns:
        print(f"    Warning: Gene '{gene_to_plot}' not found in expression data columns. Skipping.")
        continue # Skip to the next gene in the list

    # Select relevant columns and drop rows where coordinates OR the specific gene's expression is missing
    # Important: Filter based on the *current* gene_to_plot
    plot_data_gene = plot_data_base[['X', 'Y', gene_to_plot]].copy() # Use copy to avoid SettingWithCopyWarning
    plot_data_gene = plot_data_gene.dropna(subset=['X', 'Y', gene_to_plot])

    # Check if any data remains after dropping NaNs for this gene
    if plot_data_gene.empty:
        print(f"    Warning: No valid data points (non-NaN X, Y, and {gene_to_plot} expression) found for '{gene_to_plot}'. Skipping.")
        continue

    # --- Normalization specific to the current gene ---
    current_gene_expression = plot_data_gene[gene_to_plot]

    # Handle cases where expression is constant to avoid vmin=vmax error
    if current_gene_expression.min() == current_gene_expression.max():
        # If constant and non-zero, set a small range around it
        # If constant and zero, use a default range like 0-1 or slightly above 0
        if current_gene_expression.iloc[0] == 0:
             vmin = 0.0
             vmax = 1.0 # Or a small value like 0.1 if you expect low values
             print(f"    Warning: Expression for '{gene_to_plot}' is constantly 0. Using range [{vmin}, {vmax}].")
        else:
             vmin = current_gene_expression.min() - 0.1 * abs(current_gene_expression.min())
             vmax = current_gene_expression.max() + 0.1 * abs(current_gene_expression.max())
             print(f"    Warning: Expression for '{gene_to_plot}' is constant at {current_gene_expression.iloc[0]}. Using range [{vmin:.2f}, {vmax:.2f}].")

    else:
        # Use percentiles for robust normalization against outliers
        vmin = np.percentile(current_gene_expression, 1)  # 1st percentile
        vmax = np.percentile(current_gene_expression, 99)  # 99th percentile

    # Ensure vmin is strictly less than vmax after percentile calculation
    if vmin >= vmax:
       # If vmax is 0 or less, set a small positive range
       if vmax <= 0:
           vmin = 0.0
           vmax = 1.0 # Default range
           print(f"    Warning: Percentile range for '{gene_to_plot}' resulted in vmin >= vmax (<=0). Using range [{vmin}, {vmax}].")
       else:
            # Try making vmin slightly smaller than vmax
            vmin = vmax * 0.99 # or vmax - some_small_epsilon
            if vmin == vmax: # If still equal (e.g. vmax was very small)
                 vmin = 0.0 # Fallback to 0
            print(f"    Warning: Percentile range for '{gene_to_plot}' resulted in vmin >= vmax. Adjusted to [{vmin:.2f}, {vmax:.2f}].")


    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.viridis # Or cm.magma, cm.plasma, etc.

    # --- Plotting for the current gene ---
    fig, ax = plt.subplots(figsize=(10, 8)) # Adjust figsize as needed

    # Display the thumbnail image as the background
    # Use extent to match scatter plot coordinates if necessary, origin='upper' matches image convention
    ax.imshow(thumbnail, aspect='auto', extent=[0, thumbnail_width, thumbnail_height, 0])

    # Scatter plot for the current gene's expression data
    sc = ax.scatter(plot_data_gene['X'], plot_data_gene['Y'],
                    c=plot_data_gene[gene_to_plot],
                    s=100,       # Adjust marker size (smaller might be better for dense data)
                    cmap=cmap,
                    norm=norm,
                    edgecolor='none', # Optional: remove marker edges
                    alpha=0.8)      # Optional: add transparency

    # Add a colorbar
    cbar = plt.colorbar(sc, ax=ax, shrink=0.75) # Adjust shrink factor as needed
    cbar.set_label(f'{gene_to_plot} Expression Level')

    # Add title and labels
    plt.title(f'Spatial Expression of {gene_to_plot} Predicted')
    plt.xlabel('Scaled X Coordinate')
    plt.ylabel('Scaled Y Coordinate')

    # Set axis limits to match the thumbnail dimensions
    ax.set_xlim(0, thumbnail_width)
    ax.set_ylim(thumbnail_height, 0) # Flipped Y-axis (origin top-left)

    # Optional: Equal aspect ratio if spatial dimensions are meaningful
    # ax.set_aspect('equal', adjustable='box')

    # Construct the full save path using os.path.join
    # Use a safe filename (replace spaces or special chars if necessary, though gene names are often okay)
    safe_gene_name = gene_to_plot.replace('/', '_').replace('\\', '_') # Basic sanitization
    output_filename = os.path.join(save_path, f'{safe_gene_name}_Predict.png')

    # Save the figure
    try:
        plt.savefig(output_filename, dpi=200, bbox_inches='tight') # Adjust dpi, use bbox_inches='tight'
        print(f"    Plot saved to: {output_filename}")
    except Exception as e:
        print(f"    Error saving plot for {gene_to_plot}: {e}")


    # Close the plot figure to free up memory
    plt.close(fig)

print("Finished generating all plots.")

Starting plot generation for 4 genes...
  Processing: CDCA4
    Plot saved to: /sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/Results/153074A1A2A3-PPA60-APA35-MPA5/CDCA4_Predict.png
  Processing: ERRFI1
    Plot saved to: /sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/Results/153074A1A2A3-PPA60-APA35-MPA5/ERRFI1_Predict.png
  Processing: JAG1
    Plot saved to: /sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/Results/153074A1A2A3-PPA60-APA35-MPA5/JAG1_Predict.png
  Processing: XBP1
    Plot saved to: /sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/Results/153074A1A2A3-PPA60-APA35-MPA5/XBP1_Predict.png
Finished generating all plots.


# TLS annotation

In [39]:
import scanpy as sc
adata = sc.AnnData(expression_data, var=pd.DataFrame(index=gene_names))

/sibcb1/chenluonanlab8/zoujiawei/miniconda3/envs/CarfHe/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [40]:
adata.write(save_path+'adata.h5ad')

In [41]:
save_path

'/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/Results/153074A1A2A3-PPA60-APA35-MPA5/'

# spatial variable genes and cluster

In [42]:

import os,csv,re
import pandas as pd
import numpy as np
import scanpy as sc
import math
import SpaGCN as spg
from scipy.sparse import issparse
import random, torch
import warnings
warnings.filterwarnings("ignore")
import matplotlib.colors as clr
import matplotlib.pyplot as plt
import SpaGCN as spg
#In order to read in image data, we need to install some package. Here we recommend package "opencv"
#inatll opencv in python
#!pip3 install opencv-python
import cv2



In [43]:
smaple_data_path = save_path+'adata.h5ad'
adata = sc.read_h5ad(smaple_data_path)

In [44]:
img=cv2.imread(image_path)

In [45]:
file_path = valid_centers_path

In [46]:
#file_path = '/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/adjusted_ann/APA/valid_centers_standardized_150032A3B-APA85-CGP15.txt'

try:
    # 1. 读取 CSV 文件，没有表头
    spatial = pd.read_csv(file_path, header=None)

    # 2. 获取目标索引
    target_index = adata.obs.index

    # 3. --- 关键检查：确保行数匹配 ---
    if len(spatial) == len(target_index):
        print(f"CSV 行数 ({len(spatial)}) 与 adata 观测数 ({len(target_index)}) 匹配。")

        # 4. 直接将 adata.obs.index 赋值给 spatial 的索引
        spatial.index = target_index

        print("已成功将 spatial 的 index 设置为 adata.obs.index。")
        print("Spatial index 示例:", spatial.index[:5].tolist()) # 打印前 5 个索引验证

        # --- 现在你可以安全地使用 spatial 进行赋值 ---
        # 假设坐标在 CSV 的前两列（索引为 0 和 1）
        if 0 in spatial.columns and 1 in spatial.columns:
             print("Assigning spatial coordinates to adata.obs['x1'] and adata.obs['x2']...")
             adata.obs["x1"] = spatial[1] # 第 0 列
             adata.obs["x2"] = spatial[0] # 第 1 列
             print("Assignment complete.")
             print(adata.obs[['x1', 'x2']].head()) # 查看赋值结果
        else:
             print("Warning: Columns 0 and/or 1 not found in the loaded spatial data.")
             print("Available columns:", spatial.columns.tolist())

    else:
        # 如果行数不匹配，这是一个严重的问题，直接赋值索引是错误的
        print(f"错误：长度不匹配！")
        print(f"  CSV 文件 '{file_path}' 有 {len(spatial)} 行。")
        print(f"  adata.obs 有 {len(target_index)} 行。")
        print("无法直接设置索引。请检查 CSV 文件和 AnnData 对象是否对应。")
        # 你可能想在这里抛出错误或采取其他行动
        # raise ValueError("CSV row count does not match AnnData observation count.")
except FileNotFoundError:
    print(f"错误：文件未找到 - {file_path}")
except Exception as e:
    print(f"读取或处理文件时发生错误：{e}")


CSV 行数 (1717) 与 adata 观测数 (1717) 匹配。
已成功将 spatial 的 index 设置为 adata.obs.index。
Spatial index 示例: ['0', '1', '2', '3', '4']
Assigning spatial coordinates to adata.obs['x1'] and adata.obs['x2']...
Assignment complete.
     x1   x2
0   128  128
1   384  128
2   640  128
3   896  128
4  1152  128


In [47]:
adata.obs["x_array"]=adata.obs["x1"]
adata.obs["y_array"]=adata.obs["x2"]
adata.obs["x_pixel"]=adata.obs["x1"]
adata.obs["y_pixel"]=adata.obs["x2"]

In [48]:
adata.var_names=[i.upper() for i in list(adata.var_names)]
adata.var["genename"]=adata.var.index.astype("str")

In [49]:
adata.write_h5ad(save_path+"/sample_data.h5ad")

In [50]:
#Set coordinates
x_array=adata.obs["x_array"].tolist()
y_array=adata.obs["y_array"].tolist()
x_pixel=adata.obs["x_pixel"].tolist()
y_pixel=adata.obs["y_pixel"].tolist()

#Test coordinates on the image
img_new=img.copy()
for i in range(len(x_pixel)):
    x=x_pixel[i]
    y=y_pixel[i]
    img_new[int(x-20):int(x+20), int(y-20):int(y+20),:]=0

cv2.imwrite(save_path+'spatial_map.jpg', img_new)

True

In [51]:


#Calculate adjacent matrix
s=1
b=49
adj=spg.calculate_adj_matrix(x=x_pixel,y=y_pixel, x_pixel=x_pixel, y_pixel=y_pixel, image=img, beta=b, alpha=s, histology=True)
#If histlogy image is not available, SpaGCN can calculate the adjacent matrix using the fnction below
#adj=calculate_adj_matrix(x=x_pixel,y=y_pixel, histology=False)
np.savetxt(save_path+'/adj.csv', adj, delimiter=',')



Calculateing adj matrix using histology image...
Var of c0,c1,c2 =  510.52559921474875 1066.3240736035289 1201.4487214601636
Var of x,y,z =  7666675.7488420475 53581583.5416312 53581583.54163119


In [52]:


adata=sc.read(save_path+"/sample_data.h5ad")
adj=np.loadtxt(save_path+'/adj.csv', delimiter=',')
adata.var_names_make_unique()




In [53]:


p=0.5 
#Find the l value given p
l=spg.search_l(p, adj, start=0.01, end=1000, tol=0.01, max_run=100)



Run 1: l [0.01, 1000], p [0.0, 12.002488736020622]
Run 2: l [0.01, 500.005], p [0.0, 1.9360684056169006]
Run 3: l [250.0075, 500.005], p [0.2724190360465606, 1.9360684056169006]
Run 4: l [250.0075, 375.00625], p [0.2724190360465606, 0.8738319638819754]
Run 5: l [250.0075, 312.50687500000004], p [0.2724190360465606, 0.5225328471904025]
Run 6: l [281.2571875, 312.50687500000004], p [0.3857624652987892, 0.5225328471904025]
Run 7: l [296.88203125, 312.50687500000004], p [0.4511056446021333, 0.5225328471904025]
Run 8: l [304.694453125, 312.50687500000004], p [0.4860440870538951, 0.5225328471904025]
recommended l =  308.6006640625


In [54]:
#If the number of clusters known, we can use the spg.search_res() fnction to search for suitable resolution(optional)
#For this toy data, we set the number of clusters=7 since this tissue has 7 layers
n_clusters=5
#Set seed
r_seed=t_seed=n_seed=100
#Seaech for suitable resolution
res=spg.search_res(adata, adj, l, n_clusters, start=0.7, step=0.1, tol=5e-3, lr=0.05, max_epochs=20, r_seed=r_seed, t_seed=t_seed, n_seed=n_seed)

Start at res =  0.7 step =  0.1
Initializing cluster centers with louvain, resolution =  0.7
Epoch  0
Epoch  10
Res =  0.7 Num of clusters =  18
Initializing cluster centers with louvain, resolution =  0.6
Epoch  0
Epoch  10
Res =  0.6 Num of clusters =  17
Res changed to 0.6
Initializing cluster centers with louvain, resolution =  0.5
Epoch  0
Epoch  10
Res =  0.5 Num of clusters =  15
Res changed to 0.5
Initializing cluster centers with louvain, resolution =  0.4
Epoch  0
Epoch  10
Res =  0.4 Num of clusters =  13
Res changed to 0.4
Initializing cluster centers with louvain, resolution =  0.30000000000000004
Epoch  0
Epoch  10
Res =  0.30000000000000004 Num of clusters =  11
Res changed to 0.30000000000000004
Initializing cluster centers with louvain, resolution =  0.20000000000000004
Epoch  0
Epoch  10
Res =  0.20000000000000004 Num of clusters =  9
Res changed to 0.20000000000000004
Initializing cluster centers with louvain, resolution =  0.10000000000000003
Epoch  0
Epoch  10
delt

In [55]:


clf=spg.SpaGCN()
clf.set_l(l)
#Set seed
random.seed(r_seed)
torch.manual_seed(t_seed)
np.random.seed(n_seed)
#Run
clf.train(adata,adj,init_spa=True,init="louvain",res=res, tol=5e-3, lr=0.05, max_epochs=200)
y_pred, prob=clf.predict()
adata.obs["pred"]= y_pred
adata.obs["pred"]=adata.obs["pred"].astype('category')
#Do cluster refinement(optional)
#shape="hexagon" for Visium data, "square" for ST data.
adj_2d=spg.calculate_adj_matrix(x=x_array,y=y_array, histology=False)
refined_pred=spg.refine(sample_id=adata.obs.index.tolist(), pred=adata.obs["pred"].tolist(), dis=adj_2d, shape="hexagon")
adata.obs["refined_pred"]=refined_pred
adata.obs["refined_pred"]=adata.obs["refined_pred"].astype('category')
#Save results
adata.write_h5ad(save_path+"/results.h5ad")



Initializing cluster centers with louvain, resolution =  0.11250000000000003
Epoch  0
Epoch  10
delta_label  0.0023296447291788003 < tol  0.005
Reach tolerance threshold. Stopping training.
Total epoch: 19
Calculateing adj matrix using xy only...


In [56]:


adata=sc.read(save_path+"/results.h5ad")
#Set colors used
plot_color=["#F56867","#FEB915","#C798EE","#59BE86","#7495D3","#D1D1D1","#6D1A9C","#15821E","#3A84E6","#997273","#787878","#DB4C6C","#9E7A7A","#554236","#AF5F3C","#93796C","#F9BD3F","#DAB370","#877F6C","#268785"]
#Plot spatial domains
domains="pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="y_pixel",y="x_pixel",color=domains,title=domains,color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig(save_path+"/pred.png", dpi=600)
plt.close()

#Plot refined spatial domains
domains="refined_pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="y_pixel",y="x_pixel",color=domains,title=domains,color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig(save_path+"/refined_pred.png", dpi=600)
plt.close()



## SVG

In [57]:
#Read in raw data
raw=sc.read(save_path+"/sample_data.h5ad")
raw.var_names_make_unique()
raw.obs["pred"]=adata.obs["pred"].astype('category')

#Convert sparse matrix to non-sparse
raw.X=(raw.X.A if issparse(raw.X) else raw.X)
raw.raw=raw

In [58]:
raw.obs["pred"].unique()

[0, 3, 2, 1, 4]
Categories (5, int64): [0, 1, 2, 3, 4]

In [59]:
# Filtering criteria (constant for all targets)
min_in_group_fraction = 0.8
min_in_out_group_ratio = 0.5
min_fold_change = 1.5

# Target domains to iterate through
target_domains = raw.obs["pred"].unique() # 现在包含 [0,1,2,3,4,6]

# --- Start of the loop ---
print("\nStarting SVG detection loop...")
for target in target_domains:
    adj_2d = spg.calculate_adj_matrix(x=x_array, y=y_array, histology=False)
    start, end = np.quantile(adj_2d[adj_2d != 0], q=0.001), np.quantile(adj_2d[adj_2d != 0], q=0.1)
        

    r=spg.search_radius(target_cluster=target, cell_id=adata.obs.index.tolist(), x=x_array, y=y_array, pred=adata.obs["pred"].tolist(), start=start, end=end, num_min=10, num_max=14,  max_run=50)
    
    # Detect neighboring domains
    nbr_domians = spg.find_neighbor_clusters(
        target_cluster=target,
        cell_id=raw.obs.index.tolist(),
        x=raw.obs["x_array"].tolist(),
        y=raw.obs["y_array"].tolist(),
        pred=raw.obs["pred"].tolist(),
        radius=r,
        ratio=1/2
    )
    
    nbr_domians = nbr_domians[0:3]
    
    de_genes_info = spg.rank_genes_groups(
        input_adata=raw,
        target_cluster=target,
        nbr_list=nbr_domians,
        label_col="pred",
        adj_nbr=True,
        log=True
    )
    
    # Filter genes
    de_genes_info = de_genes_info[(de_genes_info["pvals_adj"] < 0.05)]
    filtered_info = de_genes_info
    filtered_info = filtered_info[
        (filtered_info["pvals_adj"] < 0.05) &
        (filtered_info["in_out_group_ratio"] > min_in_out_group_ratio) &
        (filtered_info["in_group_fraction"] > min_in_group_fraction) &
        (filtered_info["fold_change"] > min_fold_change)
    ]
    
    filtered_info = filtered_info.sort_values(by="in_group_fraction", ascending=False)
    filtered_info["target_dmain"] = target
    filtered_info["neighbors"] = str(nbr_domians)
    
    print("SVGs for domain", str(target), ":", filtered_info["genes"].tolist())
    filtered_info.to_csv(save_path+f'svgs_domain_{target}.csv', index=False)
    
    # 可视化前3个基因
    color_self = clr.LinearSegmentedColormap.from_list('pink_green', ['#3AB370',"#EAE7CC","#FD1593"], N=256)
    top3_genes = filtered_info["genes"].tolist()[:3]  # 取前3个基因
    
    for g in top3_genes:
        raw.obs["exp"] = raw.X[:, raw.var.index == g]
        ax = sc.pl.scatter(raw, alpha=1, x="y_pixel", y="x_pixel",
                          color="exp", title=g, color_map=color_self,
                          show=False, size=100000/raw.shape[0])
        ax.set_aspect('equal', 'box')
        ax.axes.invert_yaxis()
        plt.savefig(save_path + g + ".png", dpi=600)
        plt.close()
    
    print("-" * 30)

print("Loop finished.")


Starting SVG detection loop...
Calculateing adj matrix using xy only...
Calculateing adj matrix using xy only...
Calculateing adj matrix using xy only...
Run 1: radius [256.0, 2415.09912109375], num_nbr [4.544464609800363, 156.96188747731398]
Calculateing adj matrix using xy only...
Run 2: radius [256.0, 1335.549560546875], num_nbr [4.544464609800363, 61.58983666061706]
Calculateing adj matrix using xy only...
Run 3: radius [256.0, 795.7747802734375], num_nbr [4.544464609800363, 22.749546279491835]
Calculateing adj matrix using xy only...
recommended radius =  525.8873901367188 num_nbr=11.032667876588022
radius= 525.8873901367188 average number of neighbors for each spot is 11.032667876588022
 Cluster 0 has neighbors:
Dmain  3 :  1484
Dmain  2 :  171
SVGs for domain 0 : []
------------------------------
Calculateing adj matrix using xy only...
Calculateing adj matrix using xy only...
Calculateing adj matrix using xy only...
Run 1: radius [256.0, 2415.09912109375], num_nbr [4.453846153

In [60]:
save_path

'/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/1114_refine/Results/153074A1A2A3-PPA60-APA35-MPA5/'

## muti-slide integration



adata1=sc.read("/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/adjusted_ann/Results/SPA/151944A1A2A3-SPA90-APA10/sample_data.h5ad")
adata2=sc.read("/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/adjusted_ann/Results/APA/150032A3B-APA85-CGP15/sample_data.h5ad")
img1=cv2.imread("/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/adjusted_ann/SPA/standardized_151944A1A2A3-SPA90-APA10.tif")
img2=cv2.imread("/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/adjusted_ann/APA/standardized_150032A3B-APA85-CGP15.tif")



from anndata import AnnData
adata1.obs["x_pixel"]=x_pixel1
adata1.obs["y_pixel"]=y_pixel1
adata2.obs["x_pixel"]=x_pixel2-np.min(x_pixel2)+np.min(x_pixel1)
adata2.obs["y_pixel"]=y_pixel2-np.min(y_pixel2)+np.max(y_pixel1)
adata1.var_names_make_unique()
adata2.var_names_make_unique()
adata_all=AnnData.concatenate(adata1, adata2,join='inner',batch_key="dataset_batch",batch_categories=["0","1"])

adata_all.obs["z"]



X=np.array([adata_all.obs["x_pixel"], adata_all.obs["y_pixel"], adata_all.obs["z"]]).T.astype(np.float32)
adj=spg.pairwise_distance(X)



p=0.5 
#Find the l value given p
l=spg.search_l(p, adj, start=0.01, end=1000, tol=0.01, max_run=100)



res=1.0
seed=100
random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
clf=spg.SpaGCN()
clf.set_l(l)
clf.train(adata_all,adj,init_spa=True,init="louvain",res=res, tol=5e-3, lr=0.05, max_epochs=200)
y_pred, prob=clf.predict()
adata_all.obs["pred"]= y_pred
adata_all.obs["pred"]=adata_all.obs["pred"].astype('category')



save_path = '/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/adjusted_ann/Results/integration'

adata_all.write_h5ad(save_path+"/sample_data.h5ad")


colors_use=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#bec1d4', '#bb7784', '#0000ff', '#111010', '#FFFF00',   '#1f77b4', '#800080', '#959595', 
 '#7d87b9', '#bec1d4', '#d6bcc0', '#bb7784', '#8e063b', '#4a6fe3', '#8595e1', '#b5bbe3', '#e6afb9', '#e07b91', '#d33f6a', '#11c638', '#8dd593', '#c6dec7', '#ead3c6', '#f0b98d', '#ef9708', '#0fcfc0', '#9cded6', '#d5eae7', '#f3e1eb', '#f6c4e1', '#f79cd4']
num_celltype=len(adata_all.obs["pred"].unique())
adata_all.uns["pred_colors"]=list(colors_use[:num_celltype])
ax=sc.pl.scatter(adata_all,alpha=1,x="y_pixel",y="x_pixel",color="pred",show=False,size=150000/adata_all.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig(save_path+"/muti_sections_domains.png", dpi=600)
plt.close()



#If the number of clusters known, we can use the spg.search_res() fnction to search for suitable resolution(optional)
#For this toy data, we set the number of clusters=7 since this tissue has 7 layers
n_clusters=7
#Set seed
r_seed=t_seed=n_seed=100
#Seaech for suitable resolution
res=spg.search_res(adata_all, adj, l, n_clusters, start=0.7, step=0.1, tol=5e-3, lr=0.05, max_epochs=20, r_seed=r_seed, t_seed=t_seed, n_seed=n_seed)

adj_2d.shape

adata = adata_all

x_array=adata.obs["x_array"].tolist()
y_array=adata.obs["y_array"].tolist()

clf=spg.SpaGCN()
clf.set_l(l)
#Set seed
random.seed(r_seed)
torch.manual_seed(t_seed)
np.random.seed(n_seed)
#Run
clf.train(adata,adj,init_spa=True,init="louvain",res=res, tol=5e-3, lr=0.05, max_epochs=200)
y_pred, prob=clf.predict()
adata.obs["pred"]= y_pred
adata.obs["pred"]=adata.obs["pred"].astype('category')
#Do cluster refinement(optional)
#shape="hexagon" for Visium data, "square" for ST data.
adj_2d=spg.calculate_adj_matrix(x=x_array,y=y_array, histology=False)
refined_pred=spg.refine(sample_id=adata.obs.index.tolist(), pred=adata.obs["pred"].tolist(), dis=adj_2d, shape="hexagon")
adata.obs["refined_pred"]=refined_pred
adata.obs["refined_pred"]=adata.obs["refined_pred"].astype('category')
#Save results
adata.write_h5ad(save_path+"/results.h5ad")



adata=sc.read(save_path+"/results.h5ad")
#Set colors used
plot_color=["#F56867","#FEB915","#C798EE","#59BE86","#7495D3","#D1D1D1","#6D1A9C","#15821E","#3A84E6","#997273","#787878","#DB4C6C","#9E7A7A","#554236","#AF5F3C","#93796C","#F9BD3F","#DAB370","#877F6C","#268785"]
#Plot spatial domains
domains="pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="y_pixel",y="x_pixel",color=domains,title=domains,color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig(save_path+"/pred.png", dpi=600)
plt.close()

#Plot refined spatial domains
domains="refined_pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="y_pixel",y="x_pixel",color=domains,title=domains,color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig(save_path+"/refined_pred.png", dpi=600)
plt.close()


save_path

# SVG

#Read in raw data
raw=sc.read(save_path+"/sample_data.h5ad")
raw.var_names_make_unique()
raw.obs["pred"]=adata.obs["pred"].astype('category')

#Convert sparse matrix to non-sparse
raw.X=(raw.X.A if issparse(raw.X) else raw.X)
raw.raw=raw

save_path = '/sibcb1/chenluonanlab8/zoujiawei/data_lung/annotation/adjusted_ann/Results/integration/'

# Filtering criteria (constant for all targets)
min_in_group_fraction = 0.8
min_in_out_group_ratio = 0.5
min_fold_change = 1.5

# Target domains to iterate through
target_domains = [x for x in range(7) if x != 5]  # 现在包含 [0,1,2,3,4,6]

# --- Start of the loop ---
print("\nStarting SVG detection loop...")
for target in target_domains:
    adj_2d = spg.calculate_adj_matrix(x=x_array, y=y_array, histology=False)
    start, end = np.quantile(adj_2d[adj_2d != 0], q=0.001), np.quantile(adj_2d[adj_2d != 0], q=0.1)
    

    

    r = 512
    
    # Detect neighboring domains
    nbr_domians = spg.find_neighbor_clusters(
        target_cluster=target,
        cell_id=raw.obs.index.tolist(),
        x=raw.obs["x_array"].tolist(),
        y=raw.obs["y_array"].tolist(),
        pred=raw.obs["pred"].tolist(),
        radius=r,
        ratio=1/2
    )
    
    nbr_domians = nbr_domians[0:3]
    
    de_genes_info = spg.rank_genes_groups(
        input_adata=raw,
        target_cluster=target,
        nbr_list=nbr_domians,
        label_col="pred",
        adj_nbr=True,
        log=True
    )
    
    # Filter genes
    de_genes_info = de_genes_info[(de_genes_info["pvals_adj"] < 0.05)]
    filtered_info = de_genes_info
    filtered_info = filtered_info[
        (filtered_info["pvals_adj"] < 0.05) &
        (filtered_info["in_out_group_ratio"] > min_in_out_group_ratio) &
        (filtered_info["in_group_fraction"] > min_in_group_fraction) &
        (filtered_info["fold_change"] > min_fold_change)
    ]
    
    filtered_info = filtered_info.sort_values(by="in_group_fraction", ascending=False)
    filtered_info["target_dmain"] = target
    filtered_info["neighbors"] = str(nbr_domians)
    
    print("SVGs for domain", str(target), ":", filtered_info["genes"].tolist())
    filtered_info.to_csv(save_path+f'svgs_domain_{target}.csv', index=False)
    
    # 可视化前3个基因
    color_self = clr.LinearSegmentedColormap.from_list('pink_green', ['#3AB370',"#EAE7CC","#FD1593"], N=256)
    top3_genes = filtered_info["genes"].tolist()[:3]  # 取前3个基因
    
    for g in top3_genes:
        raw.obs["exp"] = raw.X[:, raw.var.index == g]
        ax = sc.pl.scatter(raw, alpha=1, x="y_pixel", y="x_pixel",
                          color="exp", title=g, color_map=color_self,
                          show=False, size=100000/raw.shape[0])
        ax.set_aspect('equal', 'box')
        ax.axes.invert_yaxis()
        plt.savefig(save_path + g + ".png", dpi=600)
        plt.close()
    
    print("-" * 30)

print("Loop finished.")